In [ ]:
# Version 8
import json
import pandas as pd
import gc
from pathlib import Path
import numpy as np

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

# The t-ohlcv data was pre-fetched
folder_path = find_project_root() / "data" / "bigData"

symbol = "BTCUSDT"
tf = "5m"

PATH_5m =folder_path / f"{symbol}-raw-{tf}.json"

with open(PATH_5m) as f:
    arr = json.load(f)

# Binance API fetched columns
cols = ["timestamp", "open", "high", "low", "close", "vol", "close-timestamp", "quote-vol", "taker#", "taker-buy-basevol","taker-buy-quotevol","checksum"]

df_5 = pd.DataFrame(arr, columns=cols)
del arr  
gc.collect() # free the Python var, preserve RAM

# Halve memory: float64 -> float32 for price/volume columns
df_5[["open", "high", "low", "close", "vol", "quote-vol", "taker#", "taker-buy-basevol","taker-buy-quotevol"]] = df_5[["open", "high", "low", "close", "vol", "quote-vol", "taker#", "taker-buy-basevol","taker-buy-quotevol"]].astype("float32")

print(f"5m Shape: {df_5.shape} | Memory: {df_5.memory_usage(deep=True).sum()/ 1e6:.2f} MB") # ~23 MB

# sort by timestamp
df_5 = df_5.sort_values('timestamp').reset_index(drop=True)
df_5.head()

5m Shape: (229771, 12) | Memory: 23.44 MB


,timestamp,open,high,low,close,vol,close-timestamp,quote-vol,taker#,taker-buy-basevol,taker-buy-quotevol,checksum
0,1703910600000,42006.898438,42026.980469,41978.000000,42014.410156,67.633591,1703910899999,2840850.500,2548.0,36.562511,1.535670e+06,0
1,1703910900000,42014.421875,42037.109375,41994.000000,42030.011719,50.828709,1703911199999,2135396.500,1963.0,27.786961,1.167242e+06,0
2,1703911200000,42030.000000,42054.800781,42029.019531,42031.988281,43.327000,1703911499999,1821481.750,1680.0,20.556490,8.641578e+05,0
3,1703911500000,42032.000000,42032.000000,42004.000000,42004.011719,24.898029,1703911799999,1046175.250,1621.0,7.909600,3.323482e+05,0
4,1703911800000,42004.000000,42004.011719,41973.921875,41973.941406,44.289219,1703912099999,1859286.375,1864.0,16.972891,7.124834e+05,0


# Does price hit ±ATR42 within the next 3 bars?
- prepare ATR42 on 5m 
- used to make triple barrier

In [ ]:
# ATR42 on 5m — equivalent to ATR14 on 15m in time

prev_close_5m = df_5['close'].shift(1)

# True range
tr_5m = pd.concat([
    df_5['high'] - df_5['low'],
    (df_5['high'] - prev_close_5m).abs(),
    (df_5['low']  - prev_close_5m).abs(),
], axis=1).max(axis=1)

# Average True Range : SMA or RMA
df_5['atr42'] = tr_5m.rolling(42).mean().astype('float32')

print(f"ATR42 range: {df_5['atr42'].min():.2f} – {df_5['atr42'].max():.2f}")
df_5[['timestamp', 'close', 'atr42']].head()

ATR42 range: 7.42 – 1552.81


,timestamp,close,atr42
229768,1772841000000,68133.273438,95.979912
229769,1772841300000,68114.023438,90.994049
229770,1772841600000,68118.156250,90.797806


# Triple Barrier
- Create ternary label on 5m

In [3]:
# triple-barrier on 5m (gap-aware)
FIVE_MIN_MS = 300_000

# Snap 5m timestamps to grid (absorbs ms jitter) — needed for gap detection
df_5['ts_5m'] = (df_5['timestamp'] // FIVE_MIN_MS) * FIVE_MIN_MS

# Triple-barrier on 5m (gap-aware)
ATR_MULT    = 1.0
MAX_BARS_5M = 3   # 1h lookahead — tune: 6=30m, 12=1h, 24=2h, 48=4h

# df_5   = ltf_df.reset_index(drop=True)

high_arr  = df_5['high'].values
low_arr   = df_5['low'].values
open_arr  = df_5['open'].values
close_arr = df_5['close'].values
atr_arr   = df_5['atr42'].values
ts5_arr   = df_5['ts_5m'].values

labels = np.full(len(df_5), np.nan)

for i in range(len(df_5)):
    if np.isnan(atr_arr[i]):
        continue

    entry = close_arr[i]
    tp    = atr_arr[i] * ATR_MULT
    sl    = atr_arr[i] * ATR_MULT
    label = 0  # default: timeout

    for k in range(1, MAX_BARS_5M + 1):
        j = i + k
        if j >= len(df_5):
            break
        # Stop at data gap — don't evaluate across market closures
        if ts5_arr[j] != ts5_arr[i] + k * FIVE_MIN_MS:
            break

        h, l, o = high_arr[j], low_arr[j], open_arr[j]
        hit_up   = (h - entry) >= tp
        hit_down = (entry - l) >= sl

        if hit_up and hit_down:
            # Both barriers in same bar: open proximity infers direction first
            label = 1 if abs(o - (entry + tp)) < abs(o - (entry - sl)) else -1
            break
        elif hit_up:
            label = 1
            break
        elif hit_down:
            label = -1
            break

    labels[i] = label

df_5['label'] = labels

# Print Stats
total  = df_5['label'].notna().sum()
n_up   = (df_5['label'] ==  1).sum()
n_down = (df_5['label'] == -1).sum()
n_time = (df_5['label'] ==  0).sum()
n_nan  = df_5['label'].isna().sum()

print(f"Total labeled  : {total:,}")
print(f"  Up   ( 1)    : {n_up:,}  ({n_up   / total * 100:.1f}%)")
print(f"  Down (-1)    : {n_down:,}  ({n_down / total * 100:.1f}%)")
print(f"  Timeout (0)  : {n_time:,}  ({n_time  / total * 100:.1f}%)  ← used Two-stage model if this Label=0 dominates") # > 50%
print(f"  NaN (warmup) : {n_nan:,}")

Total labeled  : 229,730
  Up   ( 1)    : 76,565  (33.3%)
  Down (-1)    : 78,157  (34.0%)
  Timeout (0)  : 75,008  (32.7%)  ← used Two-stage model if this Label=0 dominates
  NaN (warmup) : 41


In [4]:
# Drop no use cols
drop_cols = ["close-timestamp", "taker-buy-quotevol","checksum", "quote-vol"]
df_5.drop(columns=drop_cols, inplace=True)

# Save Data

In [5]:
# Export marker to json visulize the data on Chart
# save export to jsonl
# Export marker to json visulize the data on Chart
# save export to jsonl

save_path = find_project_root() / "data" / "mlData" / "raw"
save_path.mkdir(parents=True, exist_ok=True)

out_path = save_path / "BTCUSDT-5m-v8.jsonl"
df_5.to_json(out_path, orient="records", lines=True)
print(f"Saved {len(df_5):,} rows → {out_path}")

Saved 229,771 rows → /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/py-candle-science/data/mlData/raw/BTCUSDT-5m-v8.jsonl


In [6]:
print(df_5.columns)

Index(['timestamp', 'open', 'high', 'low', 'close', 'vol', 'taker#',
       'taker-buy-basevol', 'atr42', 'ts_5m', 'label'],
      dtype='object')
